In [1]:
import numpy as np

In [2]:
# MAtrices initialization 

In [3]:
np.random.seed(42)

In [4]:
d = 1024 # INPUT dim
k = 1024 # Output dim
r = 8 # Lora rank 
alpha = 16 # Scaling factor 

In [5]:
# Frozen base weight 
W_0 = np.random.normal(loc=0.0,scale=0.02,size=(d,k))

In [6]:
# INitialize lora matrix A
A = np.random.normal(loc=0.0,scale=0.02,size=(r,k))

In [7]:
# Lora matrix b 
B = np.zeros((d,r))

In [8]:
print(f"Base W_0 shape: {W_0.shape}")
print(f"Matrix A shape: {A.shape}")
print(f"Matrix B shape: {B.shape}")
print(f"Scaling factor (alpha/r): {alpha/r}")

Base W_0 shape: (1024, 1024)
Matrix A shape: (8, 1024)
Matrix B shape: (1024, 8)
Scaling factor (alpha/r): 2.0


# FORWARD PASS 

In [9]:
x = np.random.normal(loc=0.0,scale=1.0, size=(1,d))

# Forward pass streams 
# 1. PRETRAINED KNOWLLEDGE  
base_output = x @ W_0.T

lora_output = (alpha/r) * (x @ A.T @ B.T)

h = base_output + lora_output

In [10]:
print(f"Input 'x' shape:      {x.shape}")
print(f"Base output shape:    {base_output.shape}")
print(f"LoRA output shape:    {lora_output.shape}")
print(f"Final 'h' shape:      {h.shape}")

Input 'x' shape:      (1, 1024)
Base output shape:    (1, 1024)
LoRA output shape:    (1, 1024)
Final 'h' shape:      (1, 1024)


In [11]:
pure_base_output = x @ W_0.T

In [12]:
combined_output = pure_base_output + lora_output

In [14]:
is_identical = np.allclose(pure_base_output,combined_output)
max_diff = np.max(np.abs(pure_base_output - combined_output))

In [15]:
print(f"LoRA adapter max value at init:  {np.max(np.abs(lora_output))}")
print(f"Are the outputs byte-identical?  {is_identical}")
print(f"Maximum divergence:              {max_diff}")


LoRA adapter max value at init:  0.0
Are the outputs byte-identical?  True
Maximum divergence:              0.0


# MUlti tenant Lora 

In [16]:
A_legal = np.random.normal(loc=0.0, scale=0.05, size=(r, k))
B_legal = np.random.normal(loc=0.0, scale=0.05, size=(d, r))

In [17]:
A_medical = np.random.normal(loc=0.0, scale=0.05, size=(r, k))
B_medical = np.random.normal(loc=0.0, scale=0.05, size=(d, r))

In [18]:
h_legal = x @ W_0.T + (alpha / r) * (x @ A_legal.T @ B_legal.T)
h_medical = x @ W_0.T + (alpha / r) * (x @ A_medical.T @ B_medical.T)

In [19]:
legal_sample = h_legal.flatten()[:5]
medical_sample = h_medical.flatten()[:5]

In [20]:
is_different = not np.allclose(h_legal, h_medical)
max_divergence = np.max(np.abs(h_legal - h_medical))

In [21]:
print("=== Multi-Tenant LoRA Output ===")
print(f"Legal Tenant (First 5):   {np.round(legal_sample, 4)}")
print(f"Medical Tenant (First 5): {np.round(medical_sample, 4)}")
print("-" * 32)
print(f"Outputs successfully diverged? {is_different}")
print(f"Max divergence between tenants: {max_divergence:.4f}")

=== Multi-Tenant LoRA Output ===
Legal Tenant (First 5):   [-0.1016  1.2086 -0.3481  0.2884  0.148 ]
Medical Tenant (First 5): [-0.0148 -0.2408  0.2445 -0.0679 -0.6816]
--------------------------------
Outputs successfully diverged? True
Max divergence between tenants: 2.5496


In [22]:
# MERGED 

In [23]:
W_merged = W_0 + (alpha / r) * (B_legal @ A_legal)

In [24]:
h_merged = x @ W_merged.T

In [25]:
is_identical = np.allclose(h_legal, h_merged)
max_diff = np.max(np.abs(h_legal - h_merged))

print("=== Weight Merge Verification ===")
print(f"Original W_0 shape:        {W_0.shape}")
print(f"Merged W_merged shape:     {W_merged.shape}")
print("-" * 33)
print(f"Are the outputs identical? {is_identical}")
print(f"Maximum divergence:        {max_diff:.8e}")

=== Weight Merge Verification ===
Original W_0 shape:        (1024, 1024)
Merged W_merged shape:     (1024, 1024)
---------------------------------
Are the outputs identical? True
Maximum divergence:        2.55351296e-15
